# Query Prepared Tasks by Date and Subject

Given a subject name and a session date, list every `prepared_task` row
together with the acquisition UUID that triggered it (joined via
`task_request`).

Reads from the pathfinder forest SQLite database at
`/data/forest_la/pathfinder.forest/forest.sqlite`.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path("/Users/clairenastaskin/data/ARIA/aria.forest/forest.sqlite")

## Inputs

Prompt for the subject name (e.g. `sub-Forest001`) and the session date
(`YYYYMMDD`). The date is used to build the BIDS-style session name
`ses-YYYYMMDD` that is stored in `prepared_task.session_name`.

In [5]:
subject_name = "sub-NHS0004"
session_date = "20260316"
session_name = f"ses-{session_date}"

# subject_name = input("Subject name (e.g. sub-Forest001): ").strip()
# session_date = input("Session date (YYYYMMDD): ").strip()
# session_name = f"ses-{session_date}"

print(f"Querying subject={subject_name!r}, session={session_name!r}")

Querying subject='sub-NHS0004', session='ses-20260316'


## Query

Walk `prepared_task` → `acquisition` → `task_request` to find, for each
prepared task, the acquisition UUID that triggered it. `acquisition` links
to `prepared_task` via `acquisition.prepared_task_uuid`, and `task_request`
links back to `acquisition` via `task_request.triggered_by_acquisition_uuid`.

In [6]:
query = """
SELECT
    pt.task_label,
    pt.uuid AS prepared_task_uuid,
    tr.triggered_by_acquisition_uuid,
    TIME(pt.started_at) AS started_time,
    TIME(pt.ended_at) AS ended_time
FROM prepared_task AS pt
LEFT JOIN acquisition AS a
    ON a.prepared_task_uuid = pt.uuid
LEFT JOIN task_request AS tr
    ON tr.triggered_by_acquisition_uuid = a.uuid
WHERE pt.subject_name = ?
  AND pt.session_name = ?
ORDER BY pt.started_at
"""

# The DB lives on an exFAT volume and is in WAL mode; exFAT can't host the
# -shm file WAL needs, so a plain `mode=ro` open fails with "disk I/O error".
# `immutable=1` tells SQLite to read the main db file directly without the
# WAL/shm machinery. Only safe because the db isn't being written here.
SQLITE_URI = f"file:{DB_PATH}?mode=ro&immutable=1"

with sqlite3.connect(SQLITE_URI, uri=True) as conn:
    tasks_df = pd.read_sql_query(query, conn, params=(subject_name, session_name))

tasks_df
print(tasks_df.to_string())

                             task_label                    prepared_task_uuid triggered_by_acquisition_uuid started_time ended_time
0             task-fingersmotor_run-001  069b7c94-36b6-72bc-8000-86e3c6573695                          None         None       None
1             task-fingersmotor_run-002  069b7c99-dda7-7716-8000-45136c4485e1                          None     10:00:57   10:02:35
2             task-fingersmotor_run-002  069b7c99-dda7-7716-8000-45136c4485e1                          None     10:00:57   10:02:35
3             task-fingersmotor_run-003  069b7d59-7323-7604-8000-0efc3e91c180                          None     10:04:30   10:07:23
4            task-verbalfluency_run-001  069b7df9-a0a3-7230-8000-d660844f1f09                          None     10:50:48   10:51:34
5            task-verbalfluency_run-002  069b7e0b-b0e1-7ede-8000-0baba9596f93                          None     10:51:41   10:51:47
6            task-verbalfluency_run-003  069b7e0d-bc85-7fb5-8000-e58229f8226

## Build task_events.tsv for a single task

Given a `task_label` (e.g. `task-verbalfluency_run-002`) and the acquisition
output folder containing `logs.parquet`, emit a BIDS-style `task_events.tsv`
with columns `onset`, `duration`, `trial_type`. Onsets are seconds relative
to the first `time_ns` in `logs.parquet` (the recording start).

In [8]:
task_label = "task-verbalfluency_run-003"
parent_folder = Path("/Users/clairenastaskin/data/ARIA/subs/sub-NHS004/task-verbalfluency_run-003/")

# task_label = input("Task label (e.g. task-verbalfluency_run-002): ").strip()
# parent_folder = Path(input("Parent folder (subject-level): ").strip())

# Resolve the acquisition folder containing logs.parquet for this task_label.
# Prefer tr.triggered_by_acquisition_uuid; if None, fall back to acquisition.uuid
# joined via acquisition.prepared_task_uuid = prepared_task.uuid.
row = tasks_df[tasks_df["task_label"] == task_label]
assert len(row) == 1, f"expected one row for {task_label!r}, got {len(row)}"
row = row.iloc[0]

acq_uuid = row["triggered_by_acquisition_uuid"]
if acq_uuid is None or pd.isna(acq_uuid):
    with sqlite3.connect(SQLITE_URI, uri=True) as conn:
        fallback = pd.read_sql_query(
            "SELECT uuid FROM acquisition WHERE prepared_task_uuid = ?",
            conn,
            params=(row["prepared_task_uuid"],),
        )
    assert len(fallback) == 1, (
        f"expected one acquisition for prepared_task_uuid="
        f"{row['prepared_task_uuid']!r}, got {len(fallback)}"
    )
    acq_uuid = fallback.iloc[0]["uuid"]

output_folder = parent_folder / acq_uuid
logs_path = output_folder / "logs.parquet"
assert logs_path.exists(), f"logs.parquet not found at {logs_path}"
print(f"acquisition_uuid = {acq_uuid}")
print(f"output_folder   = {output_folder}")

acquisition_uuid = 069b7e0e-6038-745e-8000-04217b7924df
output_folder   = /Users/clairenastaskin/data/ARIA/subs/sub-NHS004/task-verbalfluency_run-003/069b7e0e-6038-745e-8000-04217b7924df


### Query stimulus screen START/STOP events for the task

`stimulus_screen.task_uuid` points at `task_request.task_uuid`, which is
linked to a `prepared_task` via the triggering acquisition. Each screen emits
`stimulus_lifetime_event` rows (`START`/`STOP`) keyed on
`stimulus_screen_event.timestamp_ns`.

In [9]:
# task_request.triggered_by_acquisition_uuid is NULL in this dataset, so the
# prepared_task -> task_request -> stimulus_task link is broken. Match screen
# events to the prepared_task by time window instead: any stimulus_screen_event
# whose timestamp_ns falls within [started_at, ended_at] belongs to this task
# (prepared_tasks within a session are sequential, not overlapping).
screen_query = """
SELECT
    ss.screen_name,
    sse.timestamp_ns,
    sle.lifetime_event
FROM prepared_task AS pt
JOIN stimulus_screen_event AS sse
    ON sse.timestamp_ns BETWEEN
        CAST((julianday(pt.started_at) - 2440587.5) * 86400 * 1000000000 AS INTEGER)
        AND CAST((julianday(pt.ended_at)   - 2440587.5) * 86400 * 1000000000 AS INTEGER)
JOIN stimulus_screen AS ss
    ON ss.uuid = sse.screen_uuid
JOIN stimulus_lifetime_event AS sle
    ON sle.screen_event_uuid = sse.uuid
WHERE pt.task_label = ?
  AND pt.subject_name = ?
  AND pt.session_name = ?
ORDER BY sse.timestamp_ns
"""

with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:
    screen_events = pd.read_sql_query(
        screen_query, conn, params=(task_label, subject_name, session_name)
    )

print(f"{len(screen_events)} screen events")
print(screen_events.head(20).to_string())

0 screen events
Empty DataFrame
Columns: [screen_name, timestamp_ns, lifetime_event]
Index: []


### Load recording reference time from `logs.parquet`

The first `time_ns` in `logs.parquet` is the wall-clock timestamp of the
first image of the recording; all event onsets are measured relative to it.

In [10]:
logs_df = pd.read_parquet(logs_path, columns=["time_ns"])
recording_start_ns = int(logs_df["time_ns"].min())
tr_seconds = float(logs_df["time_ns"].sort_values().diff().mean()) / 1e9
print(f"recording_start_ns = {recording_start_ns}")
print(f"Tr (avg inter-image interval) = {tr_seconds:.6f} s")

recording_start_ns = 1773658360942912779
Tr (avg inter-image interval) = 2.000014 s


### Build and write `task_events.tsv`

Pivot START/STOP pairs per screen into `onset` (seconds since
`recording_start_ns`), `duration` (STOP − START, seconds), and `trial_type`
(the `screen_name`).

In [11]:
starts = screen_events[screen_events["lifetime_event"] == "START"].reset_index(
    drop=True
)
stops = screen_events[screen_events["lifetime_event"] == "STOP"].reset_index(drop=True)
assert len(starts) == len(stops), "mismatched START/STOP counts"
assert (starts["screen_name"].to_numpy() == stops["screen_name"].to_numpy()).all()

events = pd.DataFrame(
    {
        "onset": (starts["timestamp_ns"] - recording_start_ns) / 1e9 + tr_seconds / 2, 
        # Set the start of the acquisition to the middle of the first TR, 
        "duration": (stops["timestamp_ns"] - starts["timestamp_ns"]) / 1e9,
        "trial_type": starts["screen_name"],
    }
)

events_path = output_folder / "task_events.tsv"
events.to_csv(events_path, sep="\t", index=False, float_format="%.6f")
print(f"wrote {events_path}")
print(events.to_string(index=False))

wrote /Users/clairenastaskin/data/ARIA/subs/sub-NHS004/task-verbalfluency_run-003/069b7e0e-6038-745e-8000-04217b7924df/task_events.tsv
Empty DataFrame
Columns: [onset, duration, trial_type]
Index: []
